# Synthetic Data Evaluation Pipeline (Demonstration)

> This notebook demonstrates the evaluation logic only and does not contain real patient data.

Minimal reproducible workflow illustrating:
- Distribution-level comparison
- Structural consistency
- Evidence-level evaluation

## 1. Import Libraries
Standard Python libraries used for data simulation and analysis.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import cohen_kappa_score

## 2. Simulated Data Construction
Synthetic "real" and "synthetic" datasets for demonstration purposes only.

In [ ]:
np.random.seed(42)

N = 200

real = pd.DataFrame({
    "bmi": np.random.normal(25, 4, N),
    "lab_tg": np.random.exponential(1.5, N),
    "lab_alt": np.random.normal(30, 10, N),
    "visceral_fat_area": np.random.normal(1000, 300, N),
    "bapwv_mean": np.random.normal(1400, 200, N),
    "tcm_fatigue": np.random.randint(0, 4, N),
    "gender": np.random.randint(0, 2, N)
})

synthetic = real.copy()
synthetic["lab_tg"] = np.clip(synthetic["lab_tg"], 0, 3)
synthetic["tcm_fatigue"] = np.random.choice([1,2], size=N)

real.head()

## 3. Distribution-Level Comparison
Kernel Density Estimation (KDE) to compare marginal distributions.

In [ ]:
plt.figure(figsize=(6,4))

sns.kdeplot(real["lab_tg"], label="Real", fill=True)
sns.kdeplot(synthetic["lab_tg"], label="Synthetic", fill=True)

plt.title("Distribution Comparison (lab_tg)")
plt.legend()
plt.show()

## 4. Structural Comparison
Correlation matrix difference to assess dependency preservation.

In [ ]:
corr_real = real.corr()
corr_syn = synthetic.corr()

diff = (corr_real - corr_syn).abs()

plt.figure(figsize=(6,5))
sns.heatmap(diff, cmap="Reds")
plt.title("Correlation Difference (|Real - Synthetic|)")
plt.show()

## 5. Evidence-Level Evaluation
Feature importance comparison using logistic regression.

In [ ]:
real["high_vfat"] = (real["visceral_fat_area"] > 1000).astype(int)
synthetic["high_vfat"] = (synthetic["visceral_fat_area"] > 1000).astype(int)

features = ["bmi", "lab_tg", "lab_alt", "tcm_fatigue"]

model_real = LogisticRegression().fit(real[features], real["high_vfat"])
model_syn = LogisticRegression().fit(synthetic[features], synthetic["high_vfat"])

coef_real = pd.Series(model_real.coef_[0], index=features)
coef_syn = pd.Series(model_syn.coef_[0], index=features)

coef_df = pd.DataFrame({
    "Real": coef_real,
    "Synthetic": coef_syn
})

coef_df.plot(kind="bar", figsize=(6,4))
plt.title("Feature Importance Comparison")
plt.show()

## 6. Model Agreement (TSTR Concept)
Comparison of predictions between real-trained and synthetic-trained models.

In [ ]:
pred_real = model_real.predict(real[features])
pred_syn = model_syn.predict(real[features])

kappa = cohen_kappa_score(pred_real, pred_syn)

print("Cohen's Kappa (Real vs Synthetic model):", kappa)